# 04 · Đồng bộ slide và transcript

**Input:** Output 01 + 02 + 03. **Thiết bị:** CPU. **Output:** 04_bundle.

Kiểm tra checksum video và phiên bản phát hiện slide của hai nhánh. Gán mỗi từ
vào interval chứa midpoint, giữ occurrence lặp lại và previous/next IDs.
Span thiếu word timing được tham chiếu chung khi qua ranh giới; không ước lượng
thời gian từng từ. Gói kết quả chứa hình ảnh, vùng crop và audio theo slide.


## 1. Môi trường

Bật **Internet** để cài thư viện/tải model. Chạy mỗi notebook trong phiên Kaggle mới, với Input và GPU như ghi ở trên.


Chỉ dùng thư viện chuẩn Python.


## 2. Hàm xử lý

Các hàm nhúng trực tiếp; không cần tải file Python đi kèm. Mở ô thu gọn để xem hoặc chỉnh thuật toán.


In [ ]:
"""Shared artifact contract. Embedded into every notebook by build_notebooks.py."""

from pathlib import Path
import hashlib
import importlib.metadata
import json
import math
import re
import shutil
import subprocess
import threading
import unicodedata
from datetime import datetime, timezone

# transformers' from_pretrained builds models under a process-wide "meta device" context
# that is not thread-safe: two loads at once (OCR thread and main thread) fail with
# "Cannot copy out of meta tensor". Every model is loaded once per session, one at a time.
MODEL_LOAD_LOCK = threading.RLock()
_SHARED_MODELS = {}


def shared_model(key, loader):
    """Load a model once per session (reused by every video) and never two at once."""
    with MODEL_LOAD_LOCK:
        if key not in _SHARED_MODELS:
            _SHARED_MODELS[key] = loader()
        return _SHARED_MODELS[key]


SCHEMA_VERSION = "lecture-rag-v1"
PIPELINE_VERSION = "l25-v1"
STAGE_NAMES = {
    1: "01_slides", 2: "02_visual", 3: "03_audio",
    4: "04_bundle", 5: "05_index", 6: "06_retrieval",
}


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    temporary.replace(path)


def fingerprint(value):
    data = json.dumps(value, ensure_ascii=False, sort_keys=True, allow_nan=False)
    return hashlib.sha256(data.encode("utf-8")).hexdigest()


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def relative_asset(root, name):
    root = Path(root).resolve()
    path = (root / name).resolve()
    if not path.is_relative_to(root):
        raise ValueError(f"Asset nằm ngoài thư mục kết quả: {name}")
    return path


def run_command(arguments):
    process = subprocess.run(
        [str(value) for value in arguments], capture_output=True, text=True,
        encoding="utf-8", errors="replace",
    )
    if process.returncode:
        raise RuntimeError(
            f"{arguments[0]} thất bại ({process.returncode}):\n{process.stderr[-3000:]}"
        )
    return process.stdout


def package_versions(names):
    versions = {}
    for name in names:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = "not-installed"
    return versions


def find_video(override=None, input_root="/kaggle/input"):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f"Không tìm thấy video: {path}")
        return path
    candidates = sorted(
        path for path in Path(input_root).rglob("*")
        if path.is_file()
        and path.suffix.lower() in {".mp4", ".mkv", ".mov", ".webm"}
        and path.stem.lower() in {"l25_v001", "tl25_v001"}
    )
    if len(candidates) != 1:
        raise ValueError(
            "Cần đúng một video L25_V001/TL25_V001. Add dataset "
            "thnhtrungnguynmif/l25a-lecture-videos-media hoặc đặt VIDEO_PATH. "
            f"Tìm thấy: {[str(path) for path in candidates]}"
        )
    return candidates[0]


def probe_video(path):
    if not shutil.which("ffprobe") or not shutil.which("ffmpeg"):
        raise RuntimeError("Cần FFmpeg và FFprobe trong PATH.")
    data = json.loads(run_command([
        "ffprobe", "-v", "error", "-show_format", "-show_streams",
        "-of", "json", str(path),
    ]))
    videos = [stream for stream in data["streams"] if stream["codec_type"] == "video"]
    audios = [stream for stream in data["streams"] if stream["codec_type"] == "audio"]
    if not videos:
        raise ValueError("Tệp không có luồng video.")
    info = data.get("format", {})
    origin = float(info.get("start_time", 0) or 0)
    duration = float(info.get("duration") or videos[0].get("duration") or 0)
    if not math.isfinite(duration) or duration <= 0:
        raise ValueError("Không đọc được thời lượng video hợp lệ.")
    return {
        "filename": Path(path).name,
        "sha256": file_sha256(path),
        "duration_sec": duration,
        "timeline_origin_sec": origin,
        "video_start_sec": float(videos[0].get("start_time", origin)),
        "audio_start_sec": float(audios[0].get("start_time", origin)) if audios else None,
        "has_audio": bool(audios),
        "width": int(videos[0]["width"]),
        "height": int(videos[0]["height"]),
        "clock": "original_presentation_timestamp_minus_format_start_time",
    }


def begin_stage(output_root, stage):
    directory = Path(output_root) / STAGE_NAMES[stage]
    directory.mkdir(parents=True, exist_ok=True)
    write_json(directory / "_RUNNING.json", {
        "stage": stage, "started_utc": datetime.now(timezone.utc).isoformat(),
    })
    return directory


def finish_stage(directory, stage, video_id, source, parameters, parents=(), extra=None):
    directory = Path(directory)
    files = {
        path.relative_to(directory).as_posix(): file_sha256(path)
        for path in sorted(directory.rglob("*"))
        if path.is_file()
        and path.name != "manifest.json"
        and not any(part.startswith("_") for part in path.relative_to(directory).parts)
        and path.suffix != ".tmp"
    }
    payload = {
        "schema_version": SCHEMA_VERSION,
        "pipeline_version": PIPELINE_VERSION,
        "stage": stage,
        "video_id": video_id,
        "source": source,
        "parameters": parameters,
        "parents": {str(parent["stage"]): parent["artifact_id"] for parent in parents},
        "files": files,
        **(extra or {}),
    }
    payload["artifact_id"] = fingerprint(payload)
    payload["created_utc"] = datetime.now(timezone.utc).isoformat()
    write_json(directory / "manifest.json", payload)
    (directory / "_RUNNING.json").unlink(missing_ok=True)
    print(f"Hoàn tất {STAGE_NAMES[stage]}: {directory}")
    return payload


def load_stage(directory, stage=None, video_id="L25_V001", verify=True):
    directory = Path(directory)
    if (directory / "_RUNNING.json").exists():
        raise ValueError(f"Bước xử lý chưa hoàn tất; cần chạy lại: {directory}")
    manifest = read_json(directory / "manifest.json")
    if manifest.get("schema_version") != SCHEMA_VERSION:
        raise ValueError(f"Schema không tương thích: {directory}")
    if stage is not None and manifest.get("stage") != stage:
        raise ValueError(f"Cần stage {stage}, nhận {manifest.get('stage')}: {directory}")
    if manifest.get("video_id") != video_id:
        raise ValueError(f"Sai video_id: {manifest.get('video_id')} != {video_id}")
    if manifest.get("pipeline_version") != PIPELINE_VERSION:
        raise ValueError(f"Sai pipeline_version: {directory}")
    signed = {key: value for key, value in manifest.items()
              if key not in {"artifact_id", "created_utc"}}
    if fingerprint(signed) != manifest["artifact_id"]:
        raise ValueError(f"Manifest đã thay đổi sau khi xuất: {directory}")
    if verify:
        for name, expected in manifest["files"].items():
            path = relative_asset(directory, name)
            if not path.is_file() or file_sha256(path) != expected:
                raise ValueError(f"Asset bị thiếu hoặc không khớp checksum: {path}")
    return manifest


def find_stage(stage, override=None, video_id="L25_V001",
               search_roots=("/kaggle/input", "/kaggle/working")):
    if override:
        directory = Path(override)
        return directory, load_stage(directory, stage, video_id)
    matches = []
    for root in map(Path, search_roots):
        if not root.exists():
            continue
        for path in root.rglob("manifest.json"):
            try:
                manifest = read_json(path)
            except (ValueError, OSError):
                continue
            if (manifest.get("schema_version") == SCHEMA_VERSION
                    and manifest.get("stage") == stage
                    and manifest.get("video_id") == video_id
                    and manifest.get("pipeline_version") == PIPELINE_VERSION):
                matches.append((path.parent, manifest))
    identities = {manifest.get("artifact_id") for _, manifest in matches}
    if not matches or len(identities) != 1:
        raise ValueError(
            f"Cần một phiên bản rõ ràng của {STAGE_NAMES[stage]}. "
            "Add Input → Notebook Output của bước trước, hoặc đặt STAGE_XX_DIR. "
            f"Các thư mục tìm được: {[str(path) for path, _ in matches]}"
        )
    directory = sorted(matches, key=lambda item: str(item[0]))[0][0]
    return directory, load_stage(directory, stage, video_id)


def require_same_source(*manifests):
    identities = {
        (item["video_id"], item["source"]["sha256"], item["pipeline_version"])
        for item in manifests
    }
    if len(identities) != 1:
        raise ValueError("Các nhánh không được tạo từ cùng video/pipeline.")


def require_parent(child, parent):
    require_same_source(child, parent)
    if child["parents"].get(str(parent["stage"])) != parent["artifact_id"]:
        raise ValueError(
            f"Stage {child['stage']} dùng phiên bản stage {parent['stage']} khác. "
            "Chạy lại nhánh với cùng kết quả phát hiện slide."
        )


def verify_intervals(intervals, duration, tolerance=0.03):
    if not intervals:
        raise ValueError("Danh sách interval rỗng.")
    ids = set()
    previous_end = 0.0
    for index, item in enumerate(intervals):
        start, end = float(item["start_sec"]), float(item["end_sec"])
        if not (math.isfinite(start) and math.isfinite(end)
                and 0 <= start < end <= duration + tolerance):
            raise ValueError(f"Interval không hợp lệ: {item}")
        if abs(start - previous_end) > tolerance:
            raise ValueError(f"Khoảng trống/chồng lấn ở interval {index}.")
        if item["segment_id"] in ids:
            raise ValueError("Trùng segment_id.")
        ids.add(item["segment_id"])
        previous_end = end
    if abs(previous_end - duration) > tolerance:
        raise ValueError("Interval cuối không bao phủ hết video.")


def owner_interval(intervals, time_sec):
    # Half-open intervals: a word exactly at a boundary belongs to the next one.
    for item in intervals:
        if item["start_sec"] <= time_sec < item["end_sec"]:
            return item
    if intervals and math.isclose(time_sec, intervals[-1]["end_sec"], abs_tol=1e-6):
        return intervals[-1]
    raise ValueError(f"Timestamp nằm ngoài lecture: {time_sec}")


def copy_stage_assets(source, destination, manifest):
    for name in manifest["files"]:
        target = relative_asset(destination, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(relative_asset(source, name), target)


def normalize_search(text):
    text = unicodedata.normalize("NFD", text.casefold().replace("đ", "d"))
    return "".join(char for char in text if not unicodedata.combining(char))


def timestamp_label(seconds):
    seconds = max(0, int(seconds))
    hours, seconds = divmod(seconds, 3600)
    minutes, seconds = divmod(seconds, 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}" if hours else f"{minutes:02}:{seconds:02}"


def unique_records(records, key):
    result = {}
    for record in records:
        if record[key] in result:
            raise ValueError(f"Trùng {key}: {record[key]}")
        result[record[key]] = record
    return result


def read_pcm_audio(path, sample_rate=16000):
    import numpy as np
    import wave

    with wave.open(str(path), "rb") as stream:
        if (stream.getframerate() != sample_rate or stream.getnchannels() != 1
                or stream.getsampwidth() != 2):
            raise ValueError("Audio phải là PCM signed 16-bit, mono, 16 kHz.")
        return np.frombuffer(stream.readframes(stream.getnframes()), dtype="<i2").copy()


def save_pcm_clip(path, pcm, start_sample, end_sample, sample_rate=16000):
    import wave

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with wave.open(str(path), "wb") as stream:
        stream.setnchannels(1)
        stream.setsampwidth(2)
        stream.setframerate(sample_rate)
        stream.writeframes(pcm[start_sample:end_sample].astype("<i2").tobytes())


In [ ]:
"""Stage-04 text policies, preserving raw OCR/ASR and word timing for review.

Conservative OCR keeps repeated lesson lines and literal layout, dropping only explicit
boilerplate. Failure guards exclude decoder loops/placeholders from evidence. Reviewed
ASR aliases require local printed support. Legacy frequency cleaning and unrestricted
correction remain available only for reproducible ablations.
"""

UNREADABLE = re.compile(r"\[(?:unreadable|non-text)\]", re.I)
# Unlimited-OCR box markup that leaks into the text, e.g. a cut-off "<|det|>header" at the end
LEAKED_TAG = re.compile(r"<\|det\|>\s*[A-Za-z_]*\s*(?:\[[\d,\s]*\]?)?|<\|/?\w{1,20}\|>")
NOISE_REGION_WORDS = ("giáo viên", "giảng viên", "người", "logo", "trang trí", "teacher", "presenter",
                      "decorative", "khuôn mặt", "chân dung")
# Onsets Whisper confuses in Vietnamese (s/x, ch/tr); a final c/t or n/ng may differ in the same
# syllable. Tone-only differences are never corrected: slide OCR drops tones more often than ASR.
ONSETS = [("tr", "ch"), ("s", "x")]
FINALS = [("ng", "n"), ("c", "t")]
OCR_FIXES = {"ǎ": "ă", "Ǎ": "Ă"}   # Unlimited-OCR writes a caron for the breve


def strip_tones(text):
    text = unicodedata.normalize("NFD", text.lower())
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    return text.replace("đ", "d")


def line_key(line):
    """Comparable form of an OCR line: no accents or punctuation (digits kept: "Câu 1" != "Câu 2")."""
    return re.sub(r"[^a-z0-9]+", "", strip_tones(line))


def repeated_lines(texts, min_share=0.5, min_slides=3):
    """Line keys that appear on at least min_share of the slides (and >= min_slides)."""
    counts, slides = {}, 0
    for text in texts:
        keys = {line_key(line) for line in (text or "").splitlines() if line_key(line)}
        if not keys:
            continue
        slides += 1
        for key in keys:
            counts[key] = counts.get(key, 0) + 1
    need = max(min_slides, math.ceil(min_share * slides))
    return {key for key, count in counts.items() if count >= need}


# Unlimited-OCR sometimes loops: L25_V005 seg 9 (lecturer + a small inset slide) came back as one
# formula line 269 times. Runs of LOOP_RUN or more are kept once; two identical answer options stay.
LOOP_RUN = 3
LOOP_PHRASE = re.compile(r"(\S.{5,}?)(?:\s*\1){%d,}" % (LOOP_RUN - 1))


def collapse_loops(lines, min_run=LOOP_RUN):
    """Keep one line of each run of >= min_run consecutive lines with the same line_key."""
    kept, index = [], 0
    while index < len(lines):
        key, end = line_key(lines[index]), index + 1
        while key and end < len(lines) and line_key(lines[end]) == key:
            end += 1
        kept.extend(lines[index:end] if end - index < min_run else lines[index:index + 1])
        index = end
    return kept


def collapse_phrase_loop(line):
    """A phrase with letters repeated back to back >= LOOP_RUN times in one line is kept once
    ("0 0 0 0" in a table row has no letters and stays)."""
    return LOOP_PHRASE.sub(lambda match: match.group(1) if re.search(r"[^\W\d_]", match.group(1))
                           else match.group(0), line)


def clean_ocr_text(text, repeated=frozenset(), conservative=False, canonical_subscripts=False):
    if conservative:
        kept = []
        for line in (text or "").splitlines():
            line = UNREADABLE.sub("", LEAKED_TAG.sub("", line)).rstrip()
            if not line.strip() or boilerplate_line(line):
                continue
            for bad, good in OCR_FIXES.items():
                line = line.replace(bad, good)
            if canonical_subscripts and not re.search(r"['\"]", line) and not re.match(
                    r"^\s*(?:\w+\s*=|(?:print|input)\s*\(|for\b|def\b|if\b|elif\b|else\b|"
                    r"while\b|return\b|import\b|from\b|#|```)", line):
                # Chemical subscripts and mathematical indices have an equivalent plain
                # digit spelling. Preserve superscript powers and all code whitespace.
                line = line.translate(str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789"))
            kept.append(line)
        return "\n".join(kept)
    kept = []
    for line in collapse_loops((text or "").splitlines()):
        line = collapse_phrase_loop(UNREADABLE.sub("", LEAKED_TAG.sub("", line))).strip()
        key = line_key(line)
        if not line or key in repeated:
            continue
        if len(line) <= 3 and not re.search(r"[A-Za-zÀ-ỹ]{2}", line):   # page numbers, bullets
            continue
        for bad, good in OCR_FIXES.items():
            line = line.replace(bad, good)
        kept.append(line)
    return "\n".join(kept)


def code_or_formula(line):
    """Protect literal syntax, including incomplete code and short symbolic lines."""
    return bool(re.search(r"[=<>+*/\\$∑Σ]|^\s*(?:print\s*\(|for\b|def\b|if\b|elif\b|else\s*:|"
                          r"while\b|return\b|import\b|from\b|#|```)", line))


def boilerplate_line(line):
    """Explicit attribution/footer rules; repetition alone is never evidence of noise.
    Bare numbers are ambiguous without trustworthy positions and therefore stay."""
    if code_or_formula(line):
        return False
    text = strip_tones(line).strip()
    return bool(re.fullmatch(r"(?:trang|page)\s+\d+(?:\s*(?:/|of)\s*\d+)?[.]?", text)
                or re.match(r"^(?:giao vien|giang vien|lecturer|copyright|©|hotline)\s*[:：]", text)
                or re.fullmatch(r"(?:logo\s+)?(?:truong (?:dai hoc quoc te sai gon|quoc te a chau)|"
                                r"saigon international university|the asian school|siu|siu[.]edu[.]vn|"
                                r"(?:bao )?thanh[- ]nien)", text))


def qwen_ocr_text(visual):
    """Recover line layout from the exported answer, without modifying stage 02."""
    raw = visual.get("raw_ocr_response")
    blocks = visual.get("ocr_blocks") or []
    if isinstance(raw, str) and raw and blocks and all(b.get("source") == "qwen3-vl" for b in blocks):
        return "\n".join(line.rstrip() for line in raw.splitlines()
                         if line.strip() and not re.match(r"^\s*\[(?:Hình|Figure)\]", line, re.I))
    return visual.get("ocr_text", "")


def heading_title(text):
    """Prefer a visible heading in the first block; body sentences and editor UI stay untitled."""
    lines = [line.strip() for line in (text or "").splitlines() if line.strip()]
    for index, line in enumerate(lines[:6]):
        if code_or_formula(line) or boilerplate_line(line):
            continue
        value = re.sub(r"^#{1,6}\s+", "", line)
        words = re.findall(r"[^\W\d_]+", value)
        if not words or len("".join(words)) < 2 or len(value) > 110 or len(words) > 16:
            continue
        if strip_tones(value).lower() in {"clear update", "run", "time", "next tip", "true", "false"}:
            continue
        letters = "".join(words)
        if len(letters) >= 3 and letters.isupper():
            return value
        if line.startswith("#") or re.match(r"^(?:Câu|Bài|Chương|Chapter|Example|Định lý|Ví dụ)\b", value):
            if len(words) <= 10 and not re.search(r"[.;!?]", value):
                return value
        if index == 0 and len(words) <= 8 and not re.search(r"[,:;.!?()|]", value):
            if sum(w[0].isupper() for w in words) >= max(1, math.ceil(len(words) * 0.6)):
                return value
    return ""


EMPTY_OCR = re.compile(r"^(?:khong co (?:chu|van ban|noi dung|text)(?: tren (?:slide|hinh|anh)(?: bai giang)?)?|"
                       r"no (?:text|readable text)(?: on (?:the )?slide)?)[.!]?$", re.I)


def text_repetition_runs(text, minimum=8, maximum_width=8):
    """Offsets of long consecutive 1-8 token phrases, excluding numeric table rows."""
    matches = list(re.finditer(r"\w+", text or ""))
    tokens = [strip_tones(m.group()) for m in matches]
    runs, index = [], 0
    while index < len(tokens):
        best = None
        for width in range(1, maximum_width + 1):
            phrase = tokens[index:index + width]
            if len(phrase) != width or not any(any(c.isalpha() for c in w) for w in phrase):
                continue
            end = index + width
            while tokens[end:end + width] == phrase:
                end += width
            repeats = (end - index) // width
            if repeats >= minimum and (best is None or end > best["end_token"]):
                best = {"phrase": " ".join(phrase), "repeats": repeats,
                        "start_token": index, "end_token": end,
                        "start_char": matches[index].start(), "end_char": matches[end - 1].end()}
        if best:
            runs.append(best)
            index = best["end_token"]
        else:
            index += 1
    return runs


def guard_ocr_text(text):
    """Remove failed output as evidence; keep raw output and reasons for a GPU retry."""
    lines = (text or "").splitlines()
    placeholders = [line for line in lines if EMPTY_OCR.fullmatch(strip_tones(line).strip())]
    text = "\n".join(line for line in lines if line not in placeholders)
    runs = text_repetition_runs(text, maximum_width=64)
    # Code/formulas can legitimately repeat; do not infer decoder failure from their tokens.
    runs = [run for run in runs if run["repeats"] >= 32
            or not code_or_formula(text[run["start_char"]:run["end_char"]])]
    removed = sum(run["end_token"] - run["start_token"] for run in runs)
    failed = bool(runs and removed / max(1, len(re.findall(r"\w+", text))) >= 0.5)
    for run in reversed(runs):
        text = text[:run["start_char"]] + text[run["end_char"]:]
    reasons = (["empty_ocr_placeholder"] if placeholders else []) + (["ocr_repetition_loop"] if runs else [])
    return ("" if failed else text.strip("\n")), {"status": "failed" if failed or (placeholders and not text.strip())
                                                    else "partial" if reasons else "available",
                                               "reasons": reasons, "loops": runs,
                                               "evidence_excluded": failed or bool(placeholders and not text.strip())}


def flag_asr_loops(words):
    """Keep every raw word/timestamp; only loop words are excluded from evidence spans."""
    tokens, owners = [], []
    for index, word in enumerate(words):
        for token in syllables(word["text"]):
            tokens.append(token)
            owners.append(index)
    runs = text_repetition_runs(" ".join(tokens))
    for run in runs:
        indices = sorted(set(owners[run["start_token"]:run["end_token"]]))
        run["word_ids"] = [words[i]["word_id"] for i in indices]
        run["start_sec"] = min(words[i]["start_sec"] for i in indices)
        run["end_sec"] = max(words[i]["end_sec"] for i in indices)
        for index in indices:
            words[index]["evidence_excluded"] = True
            words[index].setdefault("failure_reasons", []).append("asr_repetition_loop")
    return runs


# Reviewed aliases from audit 3. Canonical terms require local or repeated lecture OCR
# support. This is one global glossary, not arbitrary OCR n-grams or per-video tuning.
ASR_TERM_ALIASES = {heard: "xác suất" for heard in ("xác xuất", "sát xuất", "sắc xuất", "sát suất")}
ASR_TERM_ALIASES.update({"áp xuất": "áp suất", "nhiệt độ xôi": "nhiệt độ sôi",
                         "trí phí": "chi phí", "cao chào": "cao trào",
                         "mặt chặn": "mặt trận", "xuất điện động": "suất điện động"})


def correct_glossary_words(words, segments, lecture_support=False, code_feedback=False):
    """Atomic reviewed replacements with printed support and raw-word provenance."""
    words = [dict(word, text_asr=word.get("text_asr", word["text"])) for word in words]
    aliases = dict(ASR_TERM_ALIASES)
    if code_feedback:
        # A reviewed x/s repair for literal program feedback. Unlike technical terms,
        # it requires the local print message; lecture-wide support is insufficient.
        aliases["xai rồi"] = "sai rồi"
    support, printed_sources, feedback_words = {}, {}, set()
    for segment in segments:
        text = segment["visual"]["ocr_text_clean"].lower()
        terms = {term for term in ASR_TERM_ALIASES.values()
                 if re.search(r"(?<!\w)" + re.escape(term) + r"(?!\w)", text)}
        if code_feedback and re.search(r"\bprint\s*\([^\n]*[\"']sai roi\b", strip_tones(text)):
            terms.add("sai rồi")
            feedback_words.update(segment["speech"]["word_ids"])
        if terms:
            for term in terms:
                printed_sources.setdefault(term, []).append(segment["segment_id"])
            for word_id in segment["speech"]["word_ids"]:
                support[word_id] = (segment["segment_id"], terms)
    if lecture_support:
        # The aliases are reviewed terms, never arbitrary OCR n-grams. Require two
        # printed occurrences in the lecture before allowing non-local support.
        global_terms = {term for term, ids in printed_sources.items() if len(ids) >= 2}
        for word in words:
            segment_id, local_terms = support.get(word["word_id"], (None, set()))
            support[word["word_id"]] = (segment_id, local_terms | global_terms)
    corrections, index = [], 0
    ordered_aliases = sorted(aliases, key=lambda value: len(value.split()), reverse=True)
    while index < len(words):
        matched = False
        for alias in ordered_aliases:
            width = len(alias.split())
            window = words[index:index + width]
            if (len(window) != width or any(w.get("evidence_excluded") for w in window)
                    or not all(w["word_id"] in support for w in window)
                    or (not lecture_support and len({support[w["word_id"]][0] for w in window}) != 1)
                    or any(re.search(r"[.!?,;:]$", w["text"].strip()) for w in window[:-1])
                    or any(len(syllables(w["text"])) != 1 for w in window)):
                continue
            heard = " ".join(syllables(w["text"])[0] for w in window)
            if heard != alias:
                continue
            fixed = aliases[alias]
            if fixed == "sai rồi" and not all(w["word_id"] in feedback_words for w in window):
                continue
            if not all(fixed in support[w["word_id"]][1] for w in window):
                continue
            for word, printed in zip(window, fixed.split()):
                word["text"] = re.sub(r"\w+", lambda m: printed.capitalize() if m.group()[0].isupper() else printed,
                                      word["text"], count=1)
            corrections.append({"at_sec": window[0]["start_sec"], "heard": heard, "fixed": fixed,
                                "policy": "printed_code_feedback" if fixed == "sai rồi" else
                                          "reviewed_lecture_glossary" if lecture_support else "reviewed_glossary",
                                "support_segment_id": support[window[0]["word_id"]][0],
                                **({"printed_support_segment_ids": printed_sources[fixed]} if lecture_support else {})})
            index += width
            matched = True
            break
        if not matched:
            index += 1
    return words, corrections


def lines_new_since(previous_text, text, min_key=12):
    """Lines of text not already on the previous slide (same line_key), and the share of long
    lines that were. A slide shown again after an ad, a reveal split in two or a document
    page moved down a little repeats ~32 % of its lines on L25; the first line (title) stays."""
    seen = {line_key(line) for line in (previous_text or "").splitlines()}
    lines = (text or "").splitlines()
    long_keys = [line_key(line) for line in lines if len(line_key(line)) >= min_key]
    share = sum(key in seen for key in long_keys) / len(long_keys) if long_keys else 0.0
    kept = [line for index, line in enumerate(lines)
            if index == 0 or len(line_key(line)) < min_key or line_key(line) not in seen]
    return "\n".join(kept), round(share, 3)


def noise_region(region):
    """A figure region whose model description is only about the lecturer, a logo or decoration."""
    if region.get("kind") not in {"diagram", "figure", "image"}:
        return False
    text = f"{region.get('text') or ''} {region.get('description') or ''}".lower()
    has_numbers = bool(re.search(r"\d", region.get("text") or ""))
    return any(word in text for word in NOISE_REGION_WORDS) and not has_numbers


def clean_regions(regions):
    """Marks noise and duplicate regions (same text on the same slide) with noise=True."""
    seen = set()
    out = []
    for region in regions or []:
        region = dict(region)
        key = line_key(f"{region.get('text') or ''}{region.get('description') or ''}")
        region["noise"] = noise_region(region) or (key in seen and bool(key))
        seen.add(key)
        out.append(region)
    return out


def syllables(text):
    return re.findall(r"\w+", (text or "").lower())


def split_syllable(s):
    """(onset, rest) of a toneless syllable for the confusable onsets."""
    for onset in ("tr", "ch", "s", "x"):
        if s.startswith(onset):
            return onset, s[len(onset):]
    return "", s


def confusable(heard, printed):
    """True if a heard syllable can be a mishearing of the printed one: identical, or the onset
    is s/x or ch/tr swapped (rest equal up to tone and a c/t or n/ng final)."""
    if heard == printed:
        return True
    a, b = strip_tones(heard), strip_tones(printed)
    (oa, ra), (ob, rb) = split_syllable(a), split_syllable(b)
    if not oa or not ob or oa == ob or {oa, ob} not in ({"s", "x"}, {"ch", "tr"}):
        return False
    if ra == rb:
        return True
    for x, y in FINALS:
        for p, q in ((x, y), (y, x)):
            if ra.endswith(p) and ra[:-len(p)] + q == rb:
                return True
    return False


def ocr_vocabulary(texts, sizes=(2, 3)):
    """Counts of the syllable n-grams printed on the slides (accented, lower case)."""
    vocab = {}
    for text in texts:
        for line in clean_ocr_text(text).splitlines():
            syl = syllables(line)
            for n in sizes:
                for i in range(len(syl) - n + 1):
                    gram = tuple(syl[i:i + n])
                    if all(s.isalpha() for s in gram):
                        vocab[gram] = vocab.get(gram, 0) + 1
    return vocab


def correct_words(words, vocab, min_count=2, atomic=True):
    """Replace misheard n-grams of ASR words by the near-identical n-gram printed on the slides.
    words: [{'text', ...}] in time order. Returns (new words, corrections)."""
    words = [dict(w) for w in words]
    by_size = {}
    for gram, count in vocab.items():
        if count >= min_count:
            by_size.setdefault(len(gram), []).append(gram)
    corrections = []
    i = 0
    while i < len(words):
        done = False
        for n in ((3, 2) if atomic else (2, 3)):
            window = words[i:i + n]
            if len(window) < n:
                continue
            heard = tuple(syllables(w["text"])[0] if syllables(w["text"]) else "" for w in window)
            if "" in heard or heard in vocab:
                continue
            matches = [gram for gram in by_size.get(n, ())
                       if gram != heard and all(confusable(h, g) for h, g in zip(heard, gram))]
            if len({m for m in matches}) != 1:
                continue
            gram = matches[0]
            for word, printed in zip(window, gram):
                if syllables(word["text"])[0] != printed:
                    word.setdefault("text_asr", word["text"])
                    word["text"] = re.sub(r"\w+", lambda m: printed.capitalize() if m.group(0)[0].isupper() else printed,
                                          word["text"], count=1)
            corrections.append({"at_sec": window[0]["start_sec"], "heard": " ".join(heard), "fixed": " ".join(gram)})
            # Legacy ablations retain the greedy skip. Otherwise a compatible repair may
            # overlap this window ("thức sát xuất" -> "thức xác" -> "xác suất").
            i += 1 if atomic else n
            done = True
            break
        if not done:
            i += 1
    return words, corrections


In [ ]:
"""Ad filter: score each 1 Hz sample "lesson or ad", turn the scores into ad intervals and
cut them out of the slide intervals as their own segments (content_kind "ad").

Stage 01 calls apply_ad_filter() with the 1 Hz samples it already decoded. Later stages
skip "ad" segments: no OCR (02), their words stay with the ad (04, midpoint rule as for
slides), nothing is indexed (05).

mode "flag" (content filter phase 1): the ad detector no longer removes content. Ad time
ranges still become their own segments, but as ordinary segments with ad_flagged=True: the
OCR gate decides OCR (02), the speech is kept (04), and only the index leaves them out (05).
A guard drops every flag when the detector looks unreliable for the channel: more ad runs or
a larger share of the video than any L25 video has (L25 gold: 3-4 runs, 7.6-24.5 %).
"""

AD_LESSON_PROMPTS = [
    "a slide of a lecture presentation with text",
    "a lecture slide with a diagram, chart or formula",
    "a teacher presenting a lesson next to a slide",
    "the title slide of a lesson",
    "a page of a document or an exam with questions",
]
AD_PROMPTS = [
    "a video advertisement",
    "a promotional video of a school or university",
    "a logo animation",
    "students walking on a school campus",
    "people in a commercial",
    "a building exterior",
]


def promotional_word_spans(words, asian_spots=False):
    """Conservative CPU speech filter: explicit school spot boundaries or subscription CTAs.
    Never remove an interval by topic score alone. Mixed lesson/spot intervals retain all
    words outside the matched spot. Unbounded school mentions remain review candidates."""
    tokens, owners = [], []
    for index, word in enumerate(words):
        for token in re.findall(r"\w+", word.get("text_asr", word["text"]).lower()):
            token = unicodedata.normalize("NFD", token)
            tokens.append("".join(c for c in token if unicodedata.category(c) != "Mn").replace("đ", "d"))
            owners.append(index)
    text = " ".join(tokens)
    positions = list(re.finditer(r"\S+", text))
    ranges = []

    def add(start, end, reason):
        indices = sorted(set(owners[i] for i, p in enumerate(positions) if p.start() < end and p.end() > start))
        if not indices:
            return
        # Include a leading "Trường" in the institution name, never a teaching sentence.
        if reason == "bounded_school_spot" and indices[0] and tokens[owners.index(indices[0]) - 1] == "truong":
            indices.insert(0, indices[0] - 1)
        ranges.append({"start_sec": min(words[i]["start_sec"] for i in indices),
                       "end_sec": max(words[i]["end_sec"] for i in indices),
                       "reason": reason, "word_ids": [words[i]["word_id"] for i in indices],
                       "text": " ".join(words[i].get("text_asr", words[i]["text"]).strip() for i in indices)})

    school = re.compile(r"\b(?:dai hoc quoc (?:te|de) sai gon|truong quoc te a chau|the asian school|siu|s i u|sru)\b")
    ending = re.compile(r"\b(?:tam hon viet nam giao duc the gioi|da den luc (?:vuon|vuong) xa hon nhung gi da hoc)\b")
    asian_ending = re.compile(r"\btuong lai cua chinh minh\b")
    consumed = -1
    for start in school.finditer(text):
        if start.start() < consumed:
            continue
        asian_signature = asian_spots == "signature" and "a chau" in start.group()
        end = (asian_ending if asian_signature else ending).search(text, start.end())
        if end is None:
            continue
        inside = text[start.start():end.end()]
        if asian_signature and not re.search(
                r"\btu hao la (?:ngoi|nguoi) truong mang den nen giao duc tien tien\b",
                " ".join(inside.split()[:30])):
            continue
        # A brand plus a spot's closing slogan is a strong explicit boundary. Limit length
        # in real time so an incidental mention cannot swallow minutes of teaching.
        first = next(i for i, p in enumerate(positions) if p.end() > start.start())
        last = max(i for i, p in enumerate(positions) if p.start() < end.end())
        if words[owners[last]]["end_sec"] - words[owners[first]]["start_sec"] > 240:
            continue
        if not re.search(r"\b(?:siu|s i u|hoc phi|hoc bong|tuyen sinh|asian)\b", inside) and not (
                asian_spots and re.search(r"\btruong quoc te a chau\b", inside)):
            continue
        add(start.start(), end.end(), "bounded_school_spot")
        consumed = end.end()
    # Only the explicit call to subscribe/support is removed. QR codes, links, enrolment
    # examples and the words "trường"/"khóa học" alone are not exclusion rules.
    cta = re.compile(r"\b(?:(?:hay|nho|dung quen) )?(?:dang (?:ky|ki) kenh|ung ho kenh|kenh de ung ho|subscribe)\b")
    for match in cta.finditer(text):
        if any(r["reason"] == "bounded_school_spot" and
               words[owners[next(i for i, p in enumerate(positions) if p.end() > match.start())]]["word_id"]
               in r["word_ids"] for r in ranges):
            continue
        add(match.start(), match.end(), "subscription_cta")
    return ranges


def ad_runs(mask):
    """[start, end) index runs of a 0/1 sequence where it is 1."""
    runs, index = [], 0
    while index < len(mask):
        if mask[index]:
            end = index
            while end < len(mask) and mask[end]:
                end += 1
            runs.append([index, end])
            index = end
        else:
            index += 1
    return runs


def ad_mask(scores, threshold=0.5, smooth=5, min_ad=5, max_gap=2):
    """Per-sample ad mask: moving average, threshold, fill short gaps between two ads,
    drop ads shorter than min_ad samples except a short closing logo at the very end."""
    import numpy as np

    scores = np.asarray(scores, dtype="float64")
    if smooth > 1 and len(scores):
        # edge padding: a short closing logo is not diluted by zeros past the end of the video
        padded = np.pad(scores, (smooth // 2, smooth - 1 - smooth // 2), mode="edge")
        scores = np.convolve(padded, np.ones(smooth) / smooth, mode="valid")
    mask = (scores >= threshold).astype(int)
    for start, end in ad_runs(1 - mask):
        if 0 < start and end < len(mask) and end - start <= max_gap:
            mask[start:end] = 1
    for start, end in ad_runs(mask):
        if end - start < min_ad and not (end == len(mask) and end - start >= 2):
            mask[start:end] = 0
    return mask


def model_features(output):
    """Embedding tensor from get_image_features/get_text_features (transformers 4 or 5)."""
    import torch

    return output if torch.is_tensor(output) else output.pooler_output


def load_siglip(model_id, device="cuda"):
    """SigLIP loaded once per session (shared by the ad filter and the OCR gate).
    Returns (embed_images(images), embed_texts(texts), logit_scale, logit_bias)."""
    def load():
        import torch
        from transformers import AutoModel, AutoProcessor

        dtype = torch.float16 if str(device).startswith("cuda") else torch.float32
        model = AutoModel.from_pretrained(model_id, torch_dtype=dtype).to(device).eval()
        processor = AutoProcessor.from_pretrained(model_id)

        def embed_images(images):
            with torch.no_grad():
                pixels = processor(images=images, return_tensors="pt")["pixel_values"].to(device, dtype)
                out = model_features(model.get_image_features(pixel_values=pixels)).float()
            return torch.nn.functional.normalize(out, dim=-1).cpu().numpy()

        def embed_texts(texts):
            with torch.no_grad():
                tokens = processor(text=texts, padding="max_length", return_tensors="pt").to(device)
                out = model_features(model.get_text_features(**tokens)).float()
            return torch.nn.functional.normalize(out, dim=-1).cpu().numpy()

        return embed_images, embed_texts, model.logit_scale.exp().item(), model.logit_bias.item()

    return shared_model(("siglip", model_id, str(device)), load)


def zero_shot(embed_images, embed_texts, scale, bias, positive, negative):
    """P(positive prompts) for each image: softmax over all prompts, summed over the positive ones."""
    import numpy as np

    text = embed_texts(list(positive) + list(negative))

    def score(images):
        logits = embed_images(images) @ text.T * scale + bias
        logits = logits - logits.max(1, keepdims=True)
        probs = np.exp(logits) / np.exp(logits).sum(1, keepdims=True)
        return probs[:, :len(positive)].sum(1).tolist()

    return score


def load_ad_scorer(mode="student", device="cuda", student_path=None,
                   teacher_model="google/siglip-so400m-patch14-384"):
    """Returns (score(images) -> list of P(ad), description). mode "teacher": SigLIP
    zero-shot with the prompts above; "student": logistic regression (student_path, an
    .npz from the ad filter experiments) on the image embeddings of a small SigLIP."""
    import numpy as np

    if mode not in {"student", "teacher"}:
        raise ValueError("AD_FILTER phải là 'student', 'teacher' hoặc None.")
    student = dict(np.load(student_path, allow_pickle=False)) if mode == "student" else None
    if student and bool(student.get("with_motion", False)):
        raise ValueError("Pipeline dùng model trò chỉ có ảnh (ad_student.npz), không phải bản có chuyển động.")
    model_id = str(student["backbone"]) if student else teacher_model
    embed_images, embed_texts, scale, bias = load_siglip(model_id, device)
    if mode == "teacher":
        score = zero_shot(embed_images, embed_texts, scale, bias, AD_PROMPTS, AD_LESSON_PROMPTS)
    else:
        def score(images):
            logits = embed_images(images) @ student["coef"] + float(student["intercept"][0])
            return (1 / (1 + np.exp(-logits))).tolist()
    threshold = float(student["threshold"]) if student else 0.5
    return score, {"mode": mode, "model": model_id, "threshold": threshold,
                   "student_sha256": file_sha256(student_path) if student else None}


# OCR gate: a keyframe goes to OCR only if it shows readable lesson content. A lecturer
# talking to the camera, an intro logo or a blank screen is kept as a segment (its speech
# stays in the lesson) but is not read by the slow OCR models.
CONTENT_PROMPTS = [
    "a slide of a lecture presentation with text",
    "a lecture slide with a diagram, chart, table or formula",
    "a page of a document or an exam with questions",
    "a whiteboard or blackboard with handwriting",
    # added after the 2917-keyframe sweep (gate2): missed maths cards, code and charts
    "a chart or graph with axes and labels",
    "a scientific figure, diagram or illustration in a lesson",
    "math exercises or equations on a board or card",
    "computer code in a text editor or terminal",
]
NO_CONTENT_PROMPTS = [
    "a person talking to the camera",
    "a teacher speaking in a studio",
    "a logo or a title animation",
    "a blank screen",
    "a building or a landscape photo",
    "a group of people on a stage or outdoors",
]


def load_ocr_gate(device="cuda", model_id="google/siglip-base-patch16-224", threshold=0.10, views="frame"):
    """Returns (check(images) -> list of P(readable content), description). views: "frame"
    scores the whole keyframe; "frame+crops" also looks at its slide region, quarters and centre
    before skipping (visual.gate_keyframe), so it never skips more than "frame"."""
    if views not in {"frame", "frame+crops"}:
        raise ValueError("OCR_GATE_VIEWS phải là 'frame' hoặc 'frame+crops'.")
    embed_images, embed_texts, scale, bias = load_siglip(model_id, device)
    check = zero_shot(embed_images, embed_texts, scale, bias, CONTENT_PROMPTS, NO_CONTENT_PROMPTS)
    return check, {"model": model_id, "threshold": threshold, "views": views}


def snap(time_sec, boundaries, tolerance=1.5):
    """Move an ad edge (from 1 Hz samples) to the nearest exact state boundary of stage 01."""
    nearest = min(boundaries, key=lambda item: abs(item - time_sec))
    return nearest if abs(nearest - time_sec) <= tolerance else time_sec


def ad_time_intervals(sample_times, mask, duration, boundaries):
    intervals = []
    for start, end in ad_runs(mask):
        a = snap(sample_times[start], boundaries)
        b = duration if end >= len(sample_times) else snap(sample_times[end], boundaries)
        if intervals and a <= intervals[-1][1]:
            intervals[-1][1] = max(intervals[-1][1], b)
        elif b > a:
            intervals.append([a, b])
    return intervals


def split_ad_intervals(intervals, ads, min_interval_sec=2.0):
    """Cut ad time ranges out of slide intervals. Returns [start, end, source] pieces in time
    order; source is the index of the slide interval, or None for an ad. A lesson remnant
    shorter than min_interval_sec next to an ad is absorbed by that ad."""
    cuts = sorted({item["start_sec"] for item in intervals} | {intervals[-1]["end_sec"]}
                  | {t for a, b in ads for t in (a, b) if intervals[0]["start_sec"] < t < intervals[-1]["end_sec"]})
    pieces = []
    for start, end in zip(cuts, cuts[1:]):
        middle = (start + end) / 2
        if any(a <= middle < b for a, b in ads):
            source = None
        else:
            source = next(i for i, item in enumerate(intervals) if item["start_sec"] <= middle < item["end_sec"])
        if pieces and pieces[-1][2] == source:
            pieces[-1][1] = end
        else:
            pieces.append([start, end, source])
    changed = True
    while changed:
        changed = False
        for i, (start, end, source) in enumerate(pieces):
            next_to_ad = (i > 0 and pieces[i - 1][2] is None) or (i + 1 < len(pieces) and pieces[i + 1][2] is None)
            if source is not None and end - start < min_interval_sec and next_to_ad:
                pieces[i][2] = None
                changed = True
        merged = []
        for piece in pieces:
            if merged and merged[-1][2] is None and piece[2] is None:
                merged[-1][1] = piece[1]
            else:
                merged.append(piece)
        pieces = merged
    return pieces


def kmeans(vectors, k, iterations=25, seed=0):
    """Plain k-means on unit vectors (cosine), k-means++ start. Returns (labels, centres)."""
    import numpy as np

    rng = np.random.default_rng(seed)
    k = min(k, len(vectors))
    centres = [vectors[rng.integers(len(vectors))]]
    for _ in range(1, k):
        distance = np.clip(1 - np.max(vectors @ np.array(centres).T, axis=1), 0, None)
        centres.append(vectors[rng.choice(len(vectors), p=distance / distance.sum())] if distance.sum() > 0
                       else vectors[rng.integers(len(vectors))])
    centres = np.array(centres)
    for _ in range(iterations):
        labels = np.argmax(vectors @ centres.T, axis=1)
        for index in range(k):
            members = vectors[labels == index]
            if len(members):
                mean = members.mean(0)
                centres[index] = mean / (np.linalg.norm(mean) + 1e-9)
    return np.argmax(vectors @ centres.T, axis=1), centres


def outlier_scores(vectors, changes, *, k=6, min_share=0.05, min_runs=3, run_sec=5,
                   change_cut=0.15, weight_change=1.0, scale=1.5, offset=3.0):
    """Content filter A: P(ad) per 1 Hz sample from the video alone, no learned ad model.

    vectors: unit image embeddings (SigLIP), changes: picture change to the previous sample.
    Lecture looks = k-means clusters that hold >= min_share of the time AND come back in >=
    min_runs separate runs of >= run_sec (a lecture look recurs all over the video; an ad is a
    block). Distance to the nearest lecture centre is standardised inside the video (median /
    MAD), the share of picture cuts in a 5 s window adds weight_change, and a logistic turns
    the sum into a score for ad_mask(). Returns (scores, details)."""
    import numpy as np

    vectors = np.asarray(vectors, dtype="float32")
    vectors /= np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9
    labels, centres = kmeans(vectors, k)
    lecture = []
    for index in range(len(centres)):
        mask = labels == index
        runs = [len(run) for run in np.split(mask, np.flatnonzero(np.diff(mask.astype(int))) + 1) if run[0]]
        if mask.mean() >= min_share and sum(length >= run_sec for length in runs) >= min_runs:
            lecture.append(index)
    if not lecture:   # no recurring look: fall back to the largest cluster
        lecture = [int(np.bincount(labels).argmax())]
    distance = 1 - np.max(vectors @ centres[lecture].T, axis=1)
    median = float(np.median(distance))
    mad = float(np.median(np.abs(distance - median))) * 1.4826 or 1e-6
    z = (distance - median) / mad
    cuts = (np.asarray(changes, dtype="float32") > change_cut).astype("float32")
    cut_rate = np.convolve(cuts, np.ones(5) / 5, mode="same")
    logit = (z + weight_change * 10 * cut_rate - offset) * scale
    scores = 1 / (1 + np.exp(-logit))
    return scores.tolist(), {"lecture_clusters": len(lecture), "clusters": len(centres)}


def load_outlier_scorer(device="cuda", model_id="google/siglip-base-patch16-224", threshold=0.5, **options):
    """Ad scorer for apply_ad_filter() that compares each sample with the rest of the video
    (outlier_scores()): score(images) returns embeddings, score.video_scores(embeddings,
    records) the per-sample P(ad) once the whole video is seen."""
    import numpy as np

    embed_images, _, _, _ = load_siglip(model_id, device)

    def score(images):
        return list(embed_images(images))

    def video_scores(vectors, records):
        # stage-01 records keep the signature as uint8 (x 255)
        changes = [0.0] + [float(signature_distance(a["signature"].astype("float32") / 255.0,
                                                    b["signature"].astype("float32") / 255.0))
                           if "signature" in a and "signature" in b else 0.0
                           for a, b in zip(records, records[1:])]
        values, _ = outlier_scores(np.array(vectors), changes, **options)
        return values

    score.video_scores = video_scores
    return score, {"mode": "outlier", "model": model_id, "threshold": threshold, "options": options,
                   "student_sha256": None}


def ad_guard(ads, duration, max_runs=6, max_share=0.30):
    """Reason to distrust the ad intervals of one video, or None."""
    share = sum(b - a for a, b in ads) / max(duration, 1e-9)
    if len(ads) > max_runs:
        return f"{len(ads)} ad runs > {max_runs}"
    if share > max_share:
        return f"ads cover {share:.0%} of the video > {max_share:.0%}"
    return None


def apply_ad_filter(intervals, records, directory, video_id, duration, state_boundaries,
                    scorer, *, smooth=5, min_ad=5, max_gap=2, min_interval_sec=2.0,
                    batch_size=64, mode="cut", max_runs=6, max_share=0.30):
    """intervals: stage-01 intervals (before writing). records: the 1 Hz samples of the scan
    (time_sec, image_path in directory). mode: "cut" (ad segments, content_kind "ad") or
    "flag" (ordinary segments with ad_flagged, see the module docstring).
    Returns (new intervals, ad report)."""
    if mode not in {"cut", "flag"}:
        raise ValueError("AD_MODE phải là 'cut' hoặc 'flag'.")
    from PIL import Image

    score, info = scorer
    times, scores = [item["time_sec"] for item in records], []
    for start in range(0, len(records), batch_size):
        images = []
        for item in records[start:start + batch_size]:
            with Image.open(Path(directory) / item["image_path"]) as opened:
                images.append(opened.convert("RGB"))
        scores += score(images)
    if hasattr(score, "video_scores"):   # outlier scorer: needs the whole video at once
        scores = score.video_scores(scores, records)
    mask = ad_mask(scores, info["threshold"], smooth, min_ad, max_gap)
    ads = ad_time_intervals(times, mask, duration, sorted(set(state_boundaries)))
    pieces = split_ad_intervals(intervals, ads, min_interval_sec)
    guard = ad_guard(ads, duration, max_runs, max_share) if mode == "flag" else None

    result = []
    for start, end, source in pieces:
        number = len(result)
        if source is None:
            # representative frame: the 1 Hz sample nearest the middle of the ad
            middle = (start + end) / 2
            sample = min(records, key=lambda item: abs(item["time_sec"] - middle))
            name = f"keyframes/ad_{number:05d}.jpg"
            shutil.copyfile(Path(directory) / sample["image_path"], Path(directory) / name)
            result.append({
                "start_sec": round(start, 6), "end_sec": round(end, 6),
                "keyframe_time_sec": round(sample["time_sec"], 6), "image_path": name,
                "content_kind": "ad" if mode == "cut" else "unclassified",
                "sharpness_score": round(sample["sharpness"], 3),
                "boundary_method": "ad_filter", "reveal_states": [],
                **({"ad_flagged": guard is None} if mode == "flag" else {}),
            })
            continue
        item = dict(intervals[source])
        item["start_sec"], item["end_sec"] = round(start, 6), round(end, 6)
        if not start <= item["keyframe_time_sec"] < end:
            # the slide's own keyframe lies in the ad: use its latest state outside the ad
            inside = [state for state in item["reveal_states"]
                      if start <= state["keyframe_time_sec"] < end]
            if inside:
                state = inside[-1]
                item["keyframe_time_sec"], item["image_path"] = state["keyframe_time_sec"], state["image_path"]
            else:
                sample = min((r for r in records if start <= r["time_sec"] < end),
                             key=lambda r: -r["sharpness"], default=None)
                if sample is not None:
                    name = f"keyframes/trim_{number:05d}.jpg"
                    shutil.copyfile(Path(directory) / sample["image_path"], Path(directory) / name)
                    item["keyframe_time_sec"], item["image_path"] = sample["time_sec"], name
            item["reveal_states"] = [state for state in item["reveal_states"]
                                     if state["end_sec"] > start and state["start_sec"] < end]
        if (start, end) != (intervals[source]["start_sec"], intervals[source]["end_sec"]):
            item["boundary_method"] = item["boundary_method"] + "+ad_trim"
        if mode == "flag":
            item["ad_flagged"] = False
        result.append(item)
    for number, item in enumerate(result):
        if mode == "flag":
            inside = [value for t, value in zip(times, scores) if item["start_sec"] <= t < item["end_sec"]]
            item["ad_score"] = round(sum(inside) / len(inside), 4) if inside else None
        item["segment_id"], item["sequence"] = f"{video_id}:seg_{number:05d}", number
        item["image_sha256"] = file_sha256(Path(directory) / item["image_path"])
    report = {**info, "mode": mode, "smooth": smooth, "min_ad_sec": min_ad, "max_gap_sec": max_gap,
              **({"guard": {"max_runs": max_runs, "max_share": max_share, "triggered": guard is not None,
                            "reason": guard}} if mode == "flag" else {}),
              "ad_intervals": [[round(a, 3), round(b, 3)] for a, b in ads],
              "ad_seconds": round(sum(b - a for a, b in ads), 1),
              "sample_times": [round(value, 3) for value in times],
              "sample_scores": [round(value, 4) for value in scores]}
    return result, report


In [ ]:
"""Join the two branches without estimating slide times or inventing word timing."""


def words_to_text(words):
    text = " ".join(word["text"].strip() for word in words)
    text = re.sub(r"\s+([,.;:!?%)\]])", r"\1", text)
    return re.sub(r"([(\[])\s+", r"\1", text).strip()


def words_to_spans(words, segment_id, maximum_sec=18.0, maximum_words=70):
    spans, group = [], []

    def append_group():
        spans.append({
            "span_id": f"{segment_id}:speech_{len(spans):04d}",
            "start_sec": min(word["start_sec"] for word in group),
            "end_sec": max(word["end_sec"] for word in group),
            "text": words_to_text(group),
            "word_ids": [word["word_id"] for word in group],
            "timing_precision": "word",
        })

    for word in words:
        if group and (word["end_sec"] - group[0]["start_sec"] > maximum_sec
                      or len(group) >= maximum_words):
            append_group()
            group = []
        group.append(word)
        if re.search(r"[.!?]$", word["text"]) and len(group) >= 4:
            append_group()
            group = []
    if group:
        append_group()
    return spans


def evidence_speech_spans(words, segment_id):
    """Split at excluded words; preserve original timing and ownership for later retries."""
    spans, group = [], []
    for word in [*words, None]:
        if word is None or word.get("evidence_excluded"):
            if group:
                spans.extend(words_to_spans(group, segment_id))
                group = []
        else:
            group.append(word)
    for index, span in enumerate(spans):
        span["span_id"] = f"{segment_id}:speech_{index:04d}"
    return spans


def align_records(intervals, visuals, transcript, duration):
    verify_intervals(intervals, duration)
    visual_by_segment = unique_records(visuals, "segment_id")
    expected = {item["segment_id"] for item in intervals}
    if set(visual_by_segment) != expected:
        raise ValueError("Nhánh OCR phải có đúng một record cho mỗi visual interval.")
    words = unique_records(transcript["words"], "word_id")
    owned = {item["segment_id"]: [] for item in intervals}
    for word in sorted(words.values(), key=lambda item: (item["start_sec"], item["end_sec"])):
        start, end = word["start_sec"], word["end_sec"]
        if not (math.isfinite(start) and math.isfinite(end) and 0 <= start < end <= duration):
            raise ValueError(f"Word timing không hợp lệ: {word['word_id']}")
        interval = owner_interval(intervals, (start + end) / 2)
        owned[interval["segment_id"]].append(word)
    clips = unique_records(transcript["slide_audio_clips"], "segment_id")
    if transcript["status"] == "completed" and set(clips) != expected:
        raise ValueError("Thiếu audio clip để đối chiếu từng interval.")
    fallback = unique_records(transcript.get("fallback_spans", []), "span_id")
    for span in fallback.values():
        if not (0 <= span["start_sec"] < span["end_sec"] <= duration):
            raise ValueError(f"Fallback span không hợp lệ: {span['span_id']}")

    segments = []
    for index, interval in enumerate(intervals):
        segment_id = interval["segment_id"]
        visual = visual_by_segment[segment_id]
        spans = words_to_spans(owned[segment_id], segment_id)
        coarse_refs = [span["span_id"] for span in fallback.values()
                       if span["start_sec"] < interval["end_sec"]
                       and span["end_sec"] > interval["start_sec"]]
        segments.append({
            "segment_id": segment_id, "sequence": index,
            "start_sec": interval["start_sec"], "end_sec": interval["end_sec"],
            "previous_segment_id": intervals[index - 1]["segment_id"] if index else None,
            "next_segment_id": intervals[index + 1]["segment_id"]
            if index + 1 < len(intervals) else None,
            "content_kind": visual["content_kind"],
            # ad filter in "flag" mode: kept and read, left out of the index only (stage 05)
            **({"ad_flagged": interval["ad_flagged"], "ad_score": interval.get("ad_score")}
               if "ad_flagged" in interval else {}),
            "visual": {
                **visual,
                "image_path": interval["image_path"],
                "keyframe_time_sec": interval["keyframe_time_sec"],
            },
            "speech": {
                "text": words_to_text(owned[segment_id]), "spans": spans,
                "word_ids": [word["word_id"] for word in owned[segment_id]],
                "fallback_span_ids": coarse_refs,
                "status": "available" if spans or coarse_refs else
                ("no_audio" if transcript["status"] == "no_audio" else "no_speech_detected"),
            },
            "audio": clips.get(segment_id),
        })
    assigned = [word_id for segment in segments for word_id in segment["speech"]["word_ids"]]
    if len(assigned) != len(words) or set(assigned) != set(words):
        raise ValueError("Có từ bị mất hoặc bị gán lặp trong bước đồng bộ.")
    return segments


def word_pairs(text):
    words = re.findall(r"\w+", (text or "").lower())
    return {f"{a} {b}" for a, b in zip(words, words[1:])}


# Words that promote rather than teach. A course ad shares the subject of the lesson, so the
# topic score alone can call it relevant; these terms are counted separately (content filter B).
AD_TERMS = re.compile(r"đăng ký|ưu đãi|học phí|hotline|khóa học|khoá học|giảm giá|khuyến mãi|tuyển sinh|"
                      r"liên hệ|link|subscribe|sponsor|discount|enroll|sign up|promo", re.I)


def speech_windows(words, size=45.0, step=15.0):
    """[{start_sec, end_sec, text}] windows of the transcript words (time order)."""
    if not words:
        return []
    end, windows, start = words[-1]["end_sec"], [], 0.0
    while start < end:
        inside = [w for w in words if start <= (w["start_sec"] + w["end_sec"]) / 2 < start + size]
        if inside:
            windows.append({"start_sec": start, "end_sec": min(start + size, end), "text": words_to_text(inside)})
        start += step
    return windows


def speech_relevance(windows, embed, reference_texts=(), trim=0.25):
    """Score each speech window "belongs to this lecture": cosine to the lecture's topic centre,
    standardised inside the video (median / MAD), so no threshold is shared across channels.
    embed(texts) -> unit vectors. The centre ignores the trim share of least typical windows
    (ads, chit-chat) and also averages reference_texts (the slide text). Adds topic_score,
    topic_z and ad_terms to every window (in place) and returns them."""
    import numpy as np

    if not windows:
        return windows
    vectors = np.asarray(embed([w["text"] for w in windows]), dtype="float32")
    references = np.asarray(embed(list(reference_texts)), dtype="float32") if reference_texts else None
    centre = vectors.mean(0)
    for _ in range(2):   # re-centre without the least typical windows
        scores = vectors @ (centre / (np.linalg.norm(centre) + 1e-9))
        keep = scores >= np.quantile(scores, trim)
        pool = vectors[keep] if references is None else np.vstack([vectors[keep], references])
        centre = pool.mean(0)
    scores = vectors @ (centre / (np.linalg.norm(centre) + 1e-9))
    median = float(np.median(scores))
    mad = float(np.median(np.abs(scores - median))) * 1.4826 or 1e-6
    for window, score in zip(windows, scores):
        window["topic_score"] = round(float(score), 4)
        window["topic_z"] = round((float(score) - median) / mad, 3)
        window["ad_terms"] = len(AD_TERMS.findall(window["text"]))
    return windows


def flag_runs(values, min_run=5, max_gap=2):
    """[start, end) index runs of True after filling gaps of <= max_gap between two runs and
    dropping runs shorter than min_run samples."""
    runs, start = [], None
    for index, value in enumerate(list(values) + [False]):
        if value and start is None:
            start = index
        elif not value and start is not None:
            runs.append([start, index]); start = None
    merged = []
    for run in runs:
        if merged and run[0] - merged[-1][1] <= max_gap:
            merged[-1][1] = run[1]
        else:
            merged.append(run)
    return [run for run in merged if run[1] - run[0] >= min_run]


def slide_speech_flags(content, windows, not_slide=0.5, topic_z=-1.5, min_run=5, max_gap=2):
    """Content filter without a learned ad model: a 1 Hz sample is off-lecture when SigLIP says
    the frame shows no lecture content (1 - content score > not_slide) AND the speech around it is
    off the lecture topic (lowest topic_z of the covering speech windows < topic_z). A lecturer
    talking (no slide, on-topic speech) or a slide with silence stays lecture. L25 6 videos +
    MLS-15DA1: 92.5 % of ad seconds caught, 98.2 % / 98.9 % of lesson kept, no channel training.
    content: stage-01 content_scores.json; windows: speech_relevance() output.
    Returns [[start_sec, end_sec]] of off-lecture time."""
    times = content["times"]
    flags = []
    for time, score in zip(times, content["scores"]):
        near = [w["topic_z"] for w in windows if w["start_sec"] <= time < w["end_sec"]]
        flags.append(1 - score > not_slide and bool(near) and min(near) < topic_z)
    step = (times[-1] - times[0]) / max(1, len(times) - 1) if len(times) > 1 else 1.0
    return [[times[a], times[b - 1] + step] for a, b in flag_runs(flags, min_run, max_gap)]


def review_ad_speech(segments, threshold, min_pairs=20):
    """Stage 01 cuts ads from the picture alone; a lesson stretch it cut by mistake would lose
    its speech from the transcript and the index. An ad segment whose word pairs mostly come
    back in the rest of the lecture or on its slides is marked ad_suspect: its speech is kept.
    L25 (9 videos, 20261002 run): 27 real ads overlap 0.022-0.246; 110 runs of lesson slides
    (40-85 s) relabelled as ad overlap >= 0.207 (median 0.44). Threshold 0.25: 0/27 ads flagged,
    103/110 runs kept; 0.20: 1/27 and 110/110."""
    lesson = set()
    for segment in segments:
        if segment["content_kind"] != "ad":
            visual = segment.get("visual") or {}
            lesson |= word_pairs(segment["speech"]["text"])
            lesson |= word_pairs(visual.get("ocr_text_clean", visual.get("ocr_text", "")))
    for segment in segments:
        if segment["content_kind"] != "ad":
            continue
        pairs = word_pairs(segment["speech"]["text"])
        overlap = len(pairs & lesson) / len(pairs) if len(pairs) >= min_pairs else None
        segment["ad_review"] = {"lesson_overlap": None if overlap is None else round(overlap, 4),
                                "word_pairs": len(pairs), "threshold": threshold}
        segment["ad_suspect"] = overlap is not None and overlap >= threshold
    return segments


def run_alignment(stage01_dir, stage02_dir, stage03_dir, output_root,
                  video_id="L25_V001", ad_suspect_overlap=None, speech_embed=None, slide_speech=None,
                  conservative_ocr=False, heading_titles=False, asr_correction="legacy",
                  failure_guard=False, speech_ad_filter=False):
    """ad_suspect_overlap: None (off) or a threshold for review_ad_speech().
    speech_embed: None (off) or embed(texts) -> unit vectors (E5): every segment gets
    speech_topic = the lowest topic_z of the speech windows it overlaps (speech_relevance();
    recorded only, no decision yet) and speech_relevance.json keeps the windows.
    slide_speech: None (off) or slide_speech_flags() options ({} for the defaults): needs
    speech_embed and the stage-01 content_scores.json; a segment mostly inside off-lecture time
    gets ad_flagged (kept and read, left out of the index by stage 05)."""
    paths = [Path(stage01_dir), Path(stage02_dir), Path(stage03_dir)]
    parents = [load_stage(path, index + 1, video_id) for index, path in enumerate(paths)]
    require_parent(parents[1], parents[0])
    independent_asr = "1" not in parents[2]["parents"]
    if independent_asr:
        # Stage 03 ran in parallel with stage 01 from the same video file.
        require_same_source(parents[2], parents[0])
    else:
        require_parent(parents[2], parents[0])
    intervals = read_json(paths[0] / "intervals.json")
    visuals = read_json(paths[1] / "visual.json")
    transcript = read_json(paths[2] / "transcript.json")
    if asr_correction not in {"legacy", "off", "glossary", "glossary_lecture", "glossary_code"}:
        raise ValueError("asr_correction phải là legacy, off, glossary, glossary_lecture hoặc glossary_code.")
    if speech_ad_filter not in {False, True, "bounded_schools", "school_signatures"}:
        raise ValueError("speech_ad_filter phải là False, True, bounded_schools hoặc school_signatures.")
    if conservative_ocr not in {False, True, "canonical_subscripts"}:
        raise ValueError("conservative_ocr phải là False, True hoặc canonical_subscripts.")
    # Clean slide OCR (lines repeated on most slides, [Unreadable], lecturer/logo regions) and fix
    # lecture terms Whisper misheard using the words printed on the slides (textclean.py).
    lesson = [visual for visual in visuals if visual.get("content_kind") != "ad"]
    repeated = repeated_lines([visual.get("ocr_text", "") for visual in lesson])
    for visual in visuals:
        text = qwen_ocr_text(visual) if conservative_ocr else visual.get("ocr_text", "")
        if failure_guard:
            text, visual["ocr_quality"] = guard_ocr_text(text)
        visual["ocr_text_clean"] = clean_ocr_text(text, repeated, conservative=conservative_ocr,
                                                canonical_subscripts=conservative_ocr == "canonical_subscripts")
        if conservative_ocr:
            visual["ocr_layout_preserved"] = True
        lines = [line for line in visual["ocr_text_clean"].splitlines() if len(line.strip()) >= 12]
        visual["title_clean"] = (heading_title(visual["ocr_text_clean"]) if heading_titles
                                 and visual.get("content_kind") != "ad" else
                                 lines[0][:160] if not heading_titles and lines
                                 and visual.get("content_kind") in {"slide", "mixed", "board"} else "")
        visual["regions"] = clean_regions(visual.get("regions", []))
    previous = ""
    for visual in visuals:   # visual.json follows the interval order
        if visual.get("content_kind") == "ad":
            continue
        visual["ocr_text_new"], visual["ocr_repeat_share"] = lines_new_since(previous, visual["ocr_text_clean"])
        previous = visual["ocr_text_clean"] or previous
    vocabulary = ocr_vocabulary([visual["ocr_text_clean"] for visual in lesson])
    transcript["words"] = sorted(transcript["words"], key=lambda word: (word["start_sec"], word["end_sec"]))
    asr_loops = flag_asr_loops(transcript["words"]) if failure_guard else []
    asr_corrections = []
    if asr_correction == "legacy":
        transcript["words"], asr_corrections = correct_words(transcript["words"], vocabulary, atomic=False)
    else:
        for word in transcript["words"]:
            word.setdefault("text_asr", word["text"])
    directory = begin_stage(output_root, 4)
    audio_root = paths[2]
    if independent_asr and transcript["status"] == "completed":
        pcm = read_pcm_audio(paths[2] / "audio/lecture_16k.wav")
        transcript["slide_audio_clips"] = []
        for interval in intervals:
            start_sample = round(interval["start_sec"] * 16000)
            end_sample = min(len(pcm), round(interval["end_sec"] * 16000))
            name = f"slide_audio/seg_{interval['sequence']:05d}.wav"
            save_pcm_clip(directory / name, pcm, start_sample, end_sample)
            transcript["slide_audio_clips"].append({
                "segment_id": interval["segment_id"], "path": name,
                "start_sec": start_sample / 16000, "end_sec": end_sample / 16000,
            })
        audio_root = directory
    segments = align_records(intervals, visuals, transcript, parents[0]["source"]["duration_sec"])
    if asr_correction in {"glossary", "glossary_lecture", "glossary_code"}:
        transcript["words"], asr_corrections = correct_glossary_words(
            transcript["words"], segments, lecture_support=asr_correction in {"glossary_lecture", "glossary_code"},
            code_feedback=asr_correction == "glossary_code")
        segments = align_records(intervals, visuals, transcript, parents[0]["source"]["duration_sec"])
    word_by_id = {word["word_id"]: word for word in transcript["words"]}
    retry_segments = []
    for segment in segments:
        quality = segment["visual"].get("ocr_quality", {})
        if quality.get("reasons"):
            retry_segments.append({"video_id": video_id, "segment_id": segment["segment_id"],
                                   "start_sec": segment["start_sec"], "end_sec": segment["end_sec"],
                                   "source": "ocr", "reasons": quality["reasons"],
                                   "status": quality["status"], "loops": quality["loops"],
                                   "raw_response_path": "_inputs/02_visual/visual.json"})
        owned_ids = set(segment["speech"]["word_ids"])
        for run in asr_loops:
            if owned_ids.intersection(run["word_ids"]):
                retry_segments.append({"video_id": video_id, "segment_id": segment["segment_id"],
                                       "source": "asr", "reasons": ["asr_repetition_loop"],
                                       **run, "raw_response_path": "_inputs/03_audio/transcript.json"})
        if failure_guard:
            owned = [word_by_id[word_id] for word_id in segment["speech"]["word_ids"]]
            segment["speech"]["text_raw"] = words_to_text([{**w, "text": w.get("text_asr", w["text"])} for w in owned])
            segment["speech"]["spans"] = evidence_speech_spans(owned, segment["segment_id"])
            segment["speech"]["text"] = " ".join(span["text"] for span in segment["speech"]["spans"])
            segment["speech"]["excluded_word_ids"] = [w["word_id"] for w in owned if w.get("evidence_excluded")]
            segment["speech"]["evidence_status"] = ("partial" if segment["speech"]["spans"] else "failed") if segment["speech"]["excluded_word_ids"] else "available"
    if ad_suspect_overlap is not None:
        review_ad_speech(segments, ad_suspect_overlap)
    windows = []
    if speech_embed is not None and transcript["words"]:
        slides = [visual["ocr_text_clean"] for visual in lesson if len(visual["ocr_text_clean"]) >= 40][:40]
        windows = speech_relevance(speech_windows(transcript["words"]), speech_embed, slides)
        for segment in segments:
            near = [w for w in windows if w["start_sec"] < segment["end_sec"] and w["end_sec"] > segment["start_sec"]]
            segment["speech_topic"] = ({"z": min(w["topic_z"] for w in near),
                                        "ad_terms": sum(w["ad_terms"] for w in near)} if near else None)
    filter_report = None
    if slide_speech is not None:
        content_path = paths[0] / "content_scores.json"
        if speech_embed is None or not content_path.exists():
            raise ValueError("slide_speech cần speech_embed và content_scores.json của bước 01 (content_scorer).")
        off = slide_speech_flags(read_json(content_path), windows, **slide_speech)
        for segment in segments:
            length = max(1e-6, segment["end_sec"] - segment["start_sec"])
            inside = sum(max(0.0, min(b, segment["end_sec"]) - max(a, segment["start_sec"])) for a, b in off)
            segment["ad_flagged"] = inside / length >= 0.5
            segment["ad_score"] = round(inside / length, 3)
        filter_report = {"mode": "slide_speech", "options": slide_speech, "off_lecture": [[round(a, 2), round(b, 2)] for a, b in off],
                         "off_lecture_seconds": round(sum(b - a for a, b in off), 1),
                         "flagged_segments": sum(s["ad_flagged"] for s in segments)}
    if speech_ad_filter:
        ad_spans = promotional_word_spans(transcript["words"], asian_spots="signature" if
                                         speech_ad_filter == "school_signatures" else speech_ad_filter == "bounded_schools")
        ad_ids = {word_id for span in ad_spans for word_id in span["word_ids"]}
        for word_id in ad_ids:
            word_by_id[word_id]["evidence_excluded"] = True
            word_by_id[word_id]["promotional"] = True
        for segment in segments:
            owned = [word_by_id[word_id] for word_id in segment["speech"]["word_ids"]]
            segment["speech"]["text_raw"] = words_to_text([{**w, "text": w.get("text_asr", w["text"])} for w in owned])
            segment["speech"]["spans_before_ad_filter"] = segment["speech"]["spans"]
            segment["speech"]["spans"] = evidence_speech_spans(owned, segment["segment_id"])
            segment["speech"]["text"] = " ".join(span["text"] for span in segment["speech"]["spans"])
            segment["speech"]["promotional_word_ids"] = [w["word_id"] for w in owned if w.get("promotional")]
        filter_report = {"mode": "speech_units", "policy": "explicit_school_signatures_and_cta" if
                         speech_ad_filter == "school_signatures" else "explicit_bounded_schools_and_cta" if
                         speech_ad_filter == "bounded_schools" else "explicit_bounded_spots_and_cta",
                         "excluded_spans": ad_spans, "excluded_word_count": len(ad_ids)}
    for interval in intervals:
        name = interval["image_path"]
        target = relative_asset(directory, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(relative_asset(paths[0], name), target)
    for visual in visuals:
        if visual.get("content_image_path"):
            name = visual["content_image_path"]
            target = relative_asset(directory, name)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(relative_asset(paths[1], name), target)
        for region in visual["regions"]:
            name = region["image_path"]
            target = relative_asset(directory, name)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(relative_asset(paths[1], name), target)
    for clip in transcript["slide_audio_clips"] if audio_root != directory else ():
        name = clip["path"]
        target = relative_asset(directory, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(relative_asset(audio_root, name), target)

    write_json(directory / "segments.json", segments)
    if windows:
        write_json(directory / "speech_relevance.json", windows)
    if filter_report:
        write_json(directory / "content_filter.json", filter_report)
    write_json(directory / "asr_corrections.json", asr_corrections)
    write_json(directory / "gpu_retry_segments.json", retry_segments)
    write_json(directory / "words.json", transcript["words"])
    write_json(directory / "fallback_spans.json", transcript["fallback_spans"])
    write_json(directory / "asr_windows.json", transcript["windows"])
    # Compatible with the current app after mapping video_id to its uploaded ID.
    # Speech inside ad segments stays in segments.json but is not shown as lecture transcript,
    # unless review_ad_speech() found it sounds like the lesson (ad_suspect).
    app_spans = [span for segment in segments if segment["content_kind"] != "ad" or segment.get("ad_suspect")
                 for span in segment["speech"]["spans"]]
    app_spans += transcript["fallback_spans"]
    app_transcript = {
        "video_id": video_id, "language": transcript["language"],
        "segments": [
            {"start": span["start_sec"], "end": span["end_sec"], "text": span["text"]}
            for span in sorted(app_spans, key=lambda item: item["start_sec"])
        ],
    }
    write_json(directory / "transcript_app.json", app_transcript)
    manifest = finish_stage(
        directory, 4, video_id, parents[0]["source"],
        {"word_ownership": "midpoint_in_half_open_interval",
         "cross_boundary_fallback": "shared_reference_without_fabricated_word_times",
         "ocr_cleaning": {"repeated_lines": [] if conservative_ocr else sorted(repeated),
                          "conservative": conservative_ocr},
         "heading_titles": heading_titles, "failure_guard": failure_guard,
         "speech_ad_filter": speech_ad_filter, "asr_correction": asr_correction,
         **({"ad_suspect_overlap": ad_suspect_overlap} if ad_suspect_overlap is not None else {})},
        parents=parents,
        extra={"segment_count": len(segments), "word_count": len(transcript["words"]),
               "ad_segment_count": sum(segment["content_kind"] == "ad" for segment in segments),
               "ad_flagged_count": sum(bool(segment.get("ad_flagged")) for segment in segments),
               "asr_correction_count": len(asr_corrections),
               "noise_region_count": sum(region.get("noise", False) for visual in visuals for region in visual["regions"]),
               "ad_word_count": sum(len(segment["speech"]["word_ids"]) for segment in segments
                                    if segment["content_kind"] == "ad"),
               **({"ad_suspect_count": sum(bool(segment.get("ad_suspect")) for segment in segments)}
                  if ad_suspect_overlap is not None else {}),
               "bundle_status": "synchronized"},
    )
    return directory, manifest, segments


## 3. Cấu hình

Đường dẫn None sẽ được tìm trong Input. Nếu có nhiều phiên bản, điền đường dẫn rõ ràng để tránh chọn nhầm.


In [ ]:
from pathlib import Path

VIDEO_ID = 'L25_V001'
OUTPUT_ROOT = Path('/kaggle/working/lecture_rag') / VIDEO_ID

STAGE_01_DIR = None
STAGE_02_DIR = None
STAGE_03_DIR = None
CONSERVATIVE_OCR = 'canonical_subscripts'
HEADING_TITLES = False  # Heuristic remains opt-in; see eval/PHASE3.md
ASR_CORRECTION = 'glossary_code'
FAILURE_GUARD = True
SPEECH_AD_FILTER = 'school_signatures'


## 4. Thực thi


In [ ]:
stage01_dir, _ = find_stage(1, STAGE_01_DIR, VIDEO_ID)
stage02_dir, _ = find_stage(2, STAGE_02_DIR, VIDEO_ID)
stage03_dir, _ = find_stage(3, STAGE_03_DIR, VIDEO_ID)
output_dir, manifest, segments = run_alignment(
    stage01_dir, stage02_dir, stage03_dir, OUTPUT_ROOT, VIDEO_ID,
    conservative_ocr=CONSERVATIVE_OCR, heading_titles=HEADING_TITLES,
    asr_correction=ASR_CORRECTION, failure_guard=FAILURE_GUARD,
    speech_ad_filter=SPEECH_AD_FILTER,
)
print('Số đoạn đồng bộ:', len(segments), '| Số từ:', manifest['word_count'])


## 5. Kiểm tra / sử dụng kết quả


In [ ]:
from IPython.display import Image, display
for segment in segments[:3]:
    print(segment['segment_id'], segment['start_sec'], segment['end_sec'])
    display(Image(filename=str(output_dir / segment['visual']['image_path']), width=850))
    print('Loại:', segment['content_kind'])
    print('OCR:', segment['visual']['ocr_text'][:600])
    print('Lời giảng:', segment['speech']['text'][:1200])
    print('Trạng thái:', segment['speech']['status'])


## Tiếp theo

Save Version; notebook **05** chỉ cần Output này.
transcript_app.json tương thích format transcript backend sau khi ánh xạ
video_id sang ID lúc upload. Notebook không tự sửa dữ liệu của web app.
